In [1]:
import numpy as np
import os
from itertools import permutations, product
from collections import defaultdict
from tqdm import tqdm

# -----------------------------------
# SETTINGS
# -----------------------------------
FOLDER_PATH = "/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/polyalign/ethylene/references"
OUTPUT_FOLDER = "ethylene/gpa_ref"
TOL = 1e-6
MAX_ITER = 50

# -----------------------------------
# IO
# -----------------------------------
def read_xyz(filename):
    atoms, coords = [], []
    with open(filename) as f:
        lines = f.readlines()[2:]
        for line in lines:
            parts = line.split()
            if len(parts) < 4:
                continue
            atoms.append(parts[0])
            coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
    return np.array(atoms), np.array(coords)

def write_xyz(filename, atoms, coords, comment=""):
    with open(filename, "w") as f:
        f.write(f"{len(atoms)}\n")
        f.write(comment + "\n")
        for atom, c in zip(atoms, coords):
            f.write(f"{atom} {c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n")

# -----------------------------------
# KABSCH
# -----------------------------------
def kabsch(P, Q):
    P_cent = P - P.mean(axis=0)
    Q_cent = Q - Q.mean(axis=0)

    C = P_cent.T @ Q_cent
    V, S, Wt = np.linalg.svd(C)

    if np.linalg.det(V @ Wt) < 0:
        V[:, -1] *= -1

    U = V @ Wt
    return P_cent @ U + Q.mean(axis=0)

def rmsd(P, Q):
    return np.sqrt(((P - Q)**2).sum() / len(P))

# -----------------------------------
# Atom-type permutations
# -----------------------------------
def atom_type_permutations(atoms):
    type_indices = defaultdict(list)
    for i, atom in enumerate(atoms):
        type_indices[atom].append(i)

    type_perms = {
        t: list(permutations(idxs))
        for t, idxs in type_indices.items()
    }

    for combo in product(*type_perms.values()):
        perm_indices = [None] * len(atoms)
        for inds, perm in zip(type_indices.values(), combo):
            for orig_idx, perm_idx in zip(inds, perm):
                perm_indices[orig_idx] = perm_idx
        yield perm_indices

# -----------------------------------
# GPA with permutations
# -----------------------------------
def generalized_procrustes(coords_list, atoms,
                           tol=1e-6, max_iter=50):

    print("Generating atom-type permutations...")
    perms = list(atom_type_permutations(atoms))
    print(f"Total permutations: {len(perms)}")

    # Center initial coords
    aligned = [c - c.mean(axis=0) for c in coords_list]

    # Initial mean = first structure
    mean_shape = aligned[0].copy()

    for iteration in range(max_iter):

        print(f"\nIteration {iteration+1}")
        new_aligned = []

        total_rmsd = 0.0

        for coords in tqdm(aligned, desc="Aligning structures"):

            best_rmsd = np.inf
            best_coords = None

            for perm in perms:
                permuted = coords[perm]
                aligned_coords = kabsch(permuted, mean_shape)
                val = rmsd(aligned_coords, mean_shape)

                if val < best_rmsd:
                    best_rmsd = val
                    best_coords = aligned_coords

            total_rmsd += best_rmsd
            new_aligned.append(best_coords)

        new_mean = np.mean(new_aligned, axis=0)
        new_mean -= new_mean.mean(axis=0)

        diff = rmsd(new_mean, mean_shape)

        print(f"Mean change: {diff:.8f}")
        print(f"Average RMSD to mean: {total_rmsd/len(coords_list):.6f}")

        if diff < tol:
            print("Converged.")
            break

        mean_shape = new_mean
        aligned = new_aligned

    return new_aligned, mean_shape

# -----------------------------------
# MAIN
# -----------------------------------
def run_gpa(folder_path, output_folder):

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    files = sorted(
        [f for f in os.listdir(folder_path) if f.endswith(".xyz")]
    )

    atoms_ref = None
    coords_list = []
    filenames = []

    print("Reading XYZ files...")
    for f in files:
        atoms, coords = read_xyz(os.path.join(folder_path, f))

        if atoms_ref is None:
            atoms_ref = atoms
        elif not np.array_equal(atoms_ref, atoms):
            raise ValueError(f"Atom mismatch in {f}")

        coords_list.append(coords)
        filenames.append(f)

    aligned_coords, mean_shape = generalized_procrustes(
        coords_list,
        atoms_ref,
        tol=TOL,
        max_iter=MAX_ITER
    )

    print("\nSaving aligned structures...")
    for fname, coords in zip(filenames, aligned_coords):
        write_xyz(
            os.path.join(output_folder, f"gpa_{fname}"),
            atoms_ref,
            coords,
            comment="GPA aligned with permutation symmetry"
        )

    write_xyz(
        os.path.join(output_folder, "gpa_mean.xyz"),
        atoms_ref,
        mean_shape,
        comment="GPA mean structure"
    )

    print("Done.")

# -----------------------------------
if __name__ == "__main__":
    run_gpa(FOLDER_PATH, OUTPUT_FOLDER)


Reading XYZ files...
Generating atom-type permutations...
Total permutations: 48

Iteration 1


Aligning structures: 100%|██████████| 4/4 [00:00<00:00, 785.19it/s]


Mean change: 12.47243663
Average RMSD to mean: 12.477588

Iteration 2


Aligning structures: 100%|██████████| 4/4 [00:00<00:00, 885.72it/s]


Mean change: 0.02874925
Average RMSD to mean: 6.256068

Iteration 3


...

Mean change: 0.00002935
Average RMSD to mean: 6.255634

Iteration 7


Aligning structures: 100%|██████████| 4/4 [00:00<00:00, 723.84it/s]


Mean change: 0.00000534
Average RMSD to mean: 6.255634

Iteration 8


Aligning structures: 100%|██████████| 4/4 [00:00<00:00, 862.85it/s]

Mean change: 0.00000097
Average RMSD to mean: 6.255634
Converged.

Saving aligned structures...
Done.
